---
authors:
  - edesz
date: 2025-10-04
---

# Estimate Cohort Size Using Predicted Savings

In [ ]:
import os
from datetime import datetime
from pathlib import Path

import altair as alt
import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown

In [ ]:
_ = alt.data_transformers.disable_max_rows()
_ = alt.renderers.set_embed_options(actions=False)

In [ ]:
PROJ_ROOT = Path.cwd().parent

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

In [ ]:
import cc_churn.costs as costs
import cc_churn.costs_savings as costs_sv
import cc_churn.viz_altair as vzu
import r2.io_utils as r2io
from utils.df_utils import show_df

## About

In this notebook, we will identify the cohort of at-risk churners that should be to be targeted by the client's team.

We'll use the following

1. best ML model's prediction probabilities
2. estimate of the (net) savings from successfully targeting these customers, that is calculated from the features in the data

to identify this cohort.

### Use-Case Context

As mentioned in the project scope, the dataset provided to us by the client represents historical customer behavior where the churn outcome is already known. So, the purpose of the trained model is not to act on these historical customers, but to simulate how it would perform if deployed on similar customers in practice. By applying the model to all customers and ranking them by predicted risk of churn (`y_pred_proba`), we can estimate the financial impact if the client had proactively targeted the top-N at-risk customers at that time. The resulting analysis is therefore retrospective since we are performing it on historicla customers but forward-looking in its intent, as it estimates the expected business impact of deploying the model on future customers with similar characteristics (features) and size (~10,000).

This allows the client (credit card division manager) to determine an optimal targeting strategy under realistic assumptions about targeting (intervention) cost and effectiveness. So, the analysis in this notebook will identify this cohort by selecting the customers who most efficiently maximize the total ROI out of all ~10,000 customers.

Finally, we assume two scenarios about the client's available budget and make recommendations based for each scenario.

### Relevant Assumptions

We will use the same assumptions made during the project scoping

1. intervention (targeting) success rate of 40%
2. intervention (targeting) cost of $50 per churned customer
3. three sources of fee revenue earned from each customer based on
   -  number of credit card transacitons (assumed to be 2%)
   - credit card balance (assumed to be 18%, compared to 15-20% in Canada and 20-30% in the U.S.)
   - credit card exposure (fees determined based on category of card; see the Card_category column of the data)
4. other terms
   - loyalty discount factor of 0.9 (or 10% per year)
   - expected remaining tenure of 3 years
5. future customers behave like those characterized by this dataset of ~10,100 customers data provided to us by the client for use in this project

Please see the project scope for more details about how these assumptions are used to calcuate Customer Lifetime Value (CLV), savings and ROI. Here, a custom Python function `get_costs()` is defined in `src/cc_churn/costs.py` to estimate ROI based on customer attributes from the data and using the above assumptions.

### Outputs

Based on the deliverables [in this project's scoping document](https://github.com/edesz/credit-card-churn/blob/main/references/07_deliverables.md), this notebook produces a single file with the at-risk customers (i.e. the cohort), their features, their predicted probaility (to churn) and their business metrics will be exported to a file.

Charts will be saved as `.html` files in `reports/figures`.

## User Inputs

In [ ]:
# columns to load
columns = [
    "clientnum",
    "card_category",
    "total_revolv_bal",
    "total_trans_amt",
    "model_name",
    "y_pred_proba",
    "y_pred",
    # "best_decision_threshold",
    "is_churned",
]

# costs
# # revenue from transactions (bank earns #% of transaction volume)
interchange_rate = 0.02
# # revenue from revolving balance (~20% interest)
apr = 0.18
# # fee revenue from credit card exposure (modeled from card type)
card_fees = {"Blue": 0, "Silver": 50, "Gold": 100, "Platinum": 200}
tenure_years = 3
discount = 0.9
# # percentage of churners who can be convinced to stay (i.e. success rate
# # of saving a churning customer)
success_rate = 0.40
# # cost of intervention to get a single customer to not churn (discounts,
# # call center time, retention offers, etc.)
intervention_cost = 50
# # maximum number of customers that can be targeted based on client's budget
num_customers_max = 400

# predictions prefix
r2_key_pred = "2026-04-13/all_predictions__"

In [ ]:
reports_dir = PROJ_ROOT / "reports"
figures_dir = reports_dir / "figures"

account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
bucket_name = os.getenv("BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

# costs
multiplier = (1 - discount**tenure_years) / (1 - discount)

## Load Data with Predictions

Load predictions for all available customers

In [ ]:
%%time
df_all_pred = r2io.pandas_read_filtered_parquets_r2(
    s3_client, bucket_name, r2_key_pred, columns
).astype({"card_category": 'category', "model_name": 'category'})
print(f"Got {len(df_all_pred):,} rows of all available data")
_ = show_df(df_all_pred)
with pd.option_context('display.max_columns', None):
    display(df_all_pred.head(1))

Extract name of best ML model from model predictions

In [ ]:
best_model_name = df_all_pred["model_name"].head(1).squeeze()

## Calculate Business Metrics

Calculate business metrics from targeting all customers predicted to churn

In [ ]:
%%time
df_business_metrics = (
    df_all_pred.query("y_pred == 1")
    .pipe(
        lambda df: costs.calc_predicted_savings(
            df,
            interchange_rate=interchange_rate,
            apr=apr,
            card_fees=card_fees,
            multiplier=multiplier,
            success_rate=success_rate,
            intervention_cost=intervention_cost,
        )
    )
    .assign(
        true_savings=lambda df: np.vectorize(costs.calc_true_savings)(
            pred=df["y_pred"],
            true=df["is_churned"],
            success_rate=success_rate,
            clv=df["clv"],
            intervention_cost=intervention_cost,
        )
    )
)
print(df_business_metrics.shape)
with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
    display(df_business_metrics.head(1))

Next, we'll bin the at-risk customers based on

1. predicted probability to create risk-level bins
2. customer lifetime value to create customer value (revenue) bins
   - these are created using quantile-based revenue tiers, consistent with [ABC Analysis (Value Tiering)](https://www.dinmo.com/customer-segmentation/abc-analysis/)

In [ ]:
df_business_metrics = costs_sv.get_buckets(
    df_business_metrics,
    bins_risk_values=[0.72, 0.81, 0.9, 1.0],
    bins_risk_labels=["Low", "Medium", "High"],
    bins_clv_tier_values=[0, 0.4, 0.7, 0.9, 1.0],
    bins_clv_tier_labels=["Bronze", "Silver", "Gold", "Platinum"],
)

A custom function `save_offer()` was used to create specific *Save Offer* suggestions for actions the client's team can use to try to get customers to revert their churn decision. The offers are based on combintions of risk level and value.

## Estimate Cohort Size - Optimal Number of Customers to Target (`N`)

### Create Lookup Tool Based on Risk and Value Segments

We'll now get characteristics of each combination of risk and value bins created above

1. savings (true and predicted)
   - this will give the predicted (`expected_savings`) and true (`true_savings`) savings by acting on the predicted and true outcomes
2. number of customers
3. average predicted probability of churn
4. average CLV
5. total targeting cost (`total_intervention_cost`)
   - this is the product of the number of customers and our assumed targeting (intervention) cost per customer ($50)
6. expected savings per customer (`expected_savings_per_customer`)
   - this is a better metric than expected savings since each combination of risk and value bins contains a different number of customers

In [ ]:
df_summary_per_clv_tier_risk_level = costs_sv.summarize_campaign_mix(
    df_business_metrics,
    "expected_savings_per_customer",
    intervention_cost,
    False,
    False,
)
df_summary_per_clv_tier_risk_level

**Observations**

1. For the *Bronze / High Risk* group, the `expected_savings` is negative (-$4,691). This means it costs more to try and save these customers than they are worth. The same is true for the *Bronze* tier customers at the other risk levels (Low and Medium). The client to avoid manual outreach for *Bronze* tier customers entirely.

Next, we'll use these aggregated characteristics to create a heatmap matrix that shows these aggregated characteristics of each combination of risk level and CLV tier (revenue)

In [ ]:
text_alt_condition = alt.datum.expected_savings_per_customer > 200
tooltip = [
    alt.Tooltip("num_customers:Q", title="Number of At-Risk Customers"),
    alt.Tooltip("total_intervention_cost:Q", format=",", title="Total Cost"),
    alt.Tooltip(
        "y_pred_proba:Q", format=",.2f", title="Avg. Predicted Probability"
    ),
    alt.Tooltip("clv:Q", format=",.2f", title="Avg. CLV"),
    alt.Tooltip("expected_savings:Q", format=",.2f", title="Expected Savings"),
    alt.Tooltip(
        "savings_error_pct:Q", format=",.2f", title="Savings Error (%)"
    ),
    alt.Tooltip("save_offer:N", title="Recommendation"),
]
ptitle = alt.TitleParams(
    # text="At-Risk Priority Matrix (by CLV Tier)",
    text="At-Risk Customer Outreach & Offer Guide",
    fontSize=18,
    font="Arial",
    anchor="start",
    orient="top",
    dx=75,
    offset=10,
)

chart = vzu.plot_altair_heatmap(
    df_summary_per_clv_tier_risk_level,
    xvar="value_tier:N",
    yvar="risk_level:N",
    textvar="expected_savings_per_customer:Q",
    xsort=["Bronze", "Silver", "Gold", "Platinum"],
    ysort=["High", "Medium", "Low"],
    color_by_col="expected_savings_per_customer:Q",
    legend_title="Savings per Customer",
    border_attrs=dict(stroke="white", strokeWidth=1.0),
    scale_params=dict(scheme="reds"),
    text_fontsize=18,
    text_alt_condition=text_alt_condition,
    tooltip=tooltip,
    ptitle=ptitle,
    xtitle="Revenue Tier (CLV)",
    ytitle="Churn Risk Level",
    fig_size=dict(width=700, height=350),
    save_params=dict(
        fpath=figures_dir / "fig_28_cohort_savings_hmap_all.html"
    ),
)
chart

This heatmap shows how risk of churn intersects with customer lifetime value (revenue) levels.

The quantiles for the CLV column ensured the *top value tier* represents a truly high-priority group of customers rather than just *anyone above average*.

The client can hover over any segment and the tooltip shows exactly what incentive to offer through the *Save Offer* strategy we defined earlier. The tooltips also allow the client to quickly view all the above aggergated characteristics and the *Save Offer*s for all segments.

**Observations**

1. From the table, the *High Risk* category has the most customers. However, the most efficient way to utilize the client's budget is to target the individuals with the highest *expected_savings_per_customer*. From the chart, the
   - *Platinum / High Risk* group is the most *profitable* to save, and returns approximately 510.58 dollars per customer
   - *Platinum / Medium Risk* (at 450 dollars per capita) and *Platinum / Low Risk* (at 403 dollars per capita) groups are actually a better use of resources than the *Gold / High Risk* group (which comes in at 302 dollars per capita). So, although the latter group returns a higher predicted savings (seen from `expected_savings` in the table), the former two groups, even though they are at a lower risk level, are actually more efficient due.
2. The Platinum or Gold at High Risk are the top-priority customers (the most valuable customers who are likely to churn) that the client should target. These top-priority customers require the most aggressive targeting efforts to get them to revert their decision to cancel their credit card services at the bank.

### Estimating Cohort Size by Comparing Two Budget Scenarios

Depending on the client's budget for retention, there are two possible use-cases for this lookup tool

1. budget of at least approximately 20,000 dollars
   - requires at most 400 customers to be targeted (at an assumed cost of $50 per customer)
2. budget of at least 80,000 dollars
   - allows for up to 1,600 at-risk customers to be targeted

Based on the findings from the previous section about the poor performance of the *Bronze* revenue tier, we will exclude tiers with a negative estimated savings before extracting the recommended cohort for both scenarios.

Below is the four-step workflow to get the cohort based on scenario one, in which the client can target at most `N` customers, using the `get_cohort_within_budget()` function

1. First, we'll get top-performing combinations of `risk_level` and `value_tier` in terms of `expected_savings_per_customer` that add up to the required maximum number of customers (`num_customers_max`). Since the budget allows for up to `N` customers to be targeted, `filter_matrix_by_limit()` adds logic to ensure these combinations should capture at least `N` customers. These combinations are sorted by expected savings per customer, from highest to lowest.
2. Next, we'll get the customers that meets the required combinations of `risk_level` and `value_tier` from 1. above, starting from the top ranked combination. This approach exhausts the best-performing combinations entirely before moving down to the next-best one. Note that the customers first are ranked by `expected_savings` to ensure that even within a high-performing combination (like High-Risk / Platinum), we are picking the best candidates before moving down to the next best combination.
3. `summarize_campaign_mix()` is then called to get the breakdown of the buckets from which these top `N` customers come.
4. Finally, step 4. calculates the estimated total impact (total expected savings) and realizable benefit (average expeced savings) for the selected cohort.

Overall, this logic treats the Value / Risk segments as the primary strategic drivers. It ensures we don't chase *outlier* individual scores at the expense of targeting the most efficient overall categories. Step 2. is the most important and it ensures we are moving from the most profitable segment to our least, until the budget runs out.

#### Scenario 1 - Constrained by Intervention Budget of At Least $20,000

Below is the workflow to get the cohort for the first scenario in which at most 400 customers can be targeted

In [ ]:
%%time
df_ranked_s1, df_campaign_mix_s1, _, _ = costs_sv.get_cohort_within_budget(
    df_business_metrics,
    df_summary_per_clv_tier_risk_level.query("expected_savings > 0"),
    intervention_cost=intervention_cost,
    n=num_customers_max,
)

Below is the updated heatmap for the selected cohort

In [ ]:
df_campaign_mix_s1

In [ ]:
text_alt_condition_s1 = alt.datum.expected_savings_per_customer > 400
tooltip_s1 = [
    alt.Tooltip("num_customers:Q", title="Number of At-Risk Customers"),
    alt.Tooltip("total_intervention_cost:Q", format=",", title="Total Cost"),
    alt.Tooltip(
        "y_pred_proba:Q", format=",.2f", title="Avg. Predicted Probability"
    ),
    alt.Tooltip("clv:Q", format=",.2f", title="Avg. CLV"),
    alt.Tooltip("expected_savings:Q", format=",.2f", title="Expected Savings"),
    alt.Tooltip(
        "savings_error_pct:Q", format=",.2f", title="Savings Error (%)"
    ),
    alt.Tooltip("save_offer:N", title="Recommendation"),
]
ptitle_s1 = alt.TitleParams(
    # text="At-Risk Priority Matrix (by CLV Tier)",
    text="At-Risk Cohort Outreach & Offer Guide",
    fontSize=18,
    font="Arial",
    anchor="start",
    orient="top",
    dx=75,
    offset=10,
)

chart_budget = vzu.plot_altair_heatmap(
    df_campaign_mix_s1,
    xvar="value_tier:N",
    yvar="risk_level:N",
    textvar="expected_savings_per_customer:Q",
    xsort=["Bronze", "Silver", "Gold", "Platinum"],
    ysort=["High", "Medium", "Low"],
    color_by_col="expected_savings_per_customer:Q",
    border_attrs=dict(stroke="white", strokeWidth=1.0),
    scale_params=dict(scheme="reds"),
    text_fontsize=18,
    text_alt_condition=text_alt_condition_s1,
    tooltip=tooltip_s1,
    ptitle=ptitle_s1,
    legend_title="Savings per Customer",
    xtitle="Revenue Tier (CLV)",
    ytitle="Churn Risk Level",
    fig_size=dict(width=300, height=350),
    save_params=dict(
        fpath=figures_dir / "fig_29_cohort_savings_hmap_cs1.html"
    ),
)
chart_budget

#### Scenario 2 - Higher Budget of At Least $80,000

For the second scenario a higher budget is available that allows for targeting all customers predicted to be at risk of churning, so we recommend the client target all at-risk customers, excluding those in a tier with negative estimated savings.

Below is the workflow to get the cohort for this scenario

In [ ]:
%%time
df_ranked_s2, df_campaign_mix_s2, _, _ = costs_sv.get_cohort_within_budget(
    df_business_metrics,
    df_summary_per_clv_tier_risk_level.query("expected_savings > 0"),
    intervention_cost=intervention_cost,
    n=len(df_business_metrics),
)

Similar to the first scenario, below is the updated heatmap for the selected cohort

In [ ]:
text_alt_condition_s2 = alt.datum.expected_savings_per_customer > 200
tooltip_s2 = [
    alt.Tooltip("num_customers:Q", title="Number of At-Risk Customers"),
    alt.Tooltip("total_intervention_cost:Q", format=",", title="Total Cost"),
    alt.Tooltip(
        "y_pred_proba:Q", format=",.2f", title="Avg. Predicted Probability"
    ),
    alt.Tooltip("clv:Q", format=",.2f", title="Avg. CLV"),
    alt.Tooltip("expected_savings:Q", format=",.2f", title="Expected Savings"),
    alt.Tooltip(
        "savings_error_pct:Q", format=",.2f", title="Savings Error (%)"
    ),
    alt.Tooltip("save_offer:N", title="Recommendation"),
]
ptitle_s2 = alt.TitleParams(
    # text="At-Risk Priority Matrix (by CLV Tier)",
    text="At-Risk Cohort Outreach & Offer Guide",
    fontSize=18,
    font="Arial",
    anchor="start",
    orient="top",
    dx=75,
    offset=10,
)

chart_budget = vzu.plot_altair_heatmap(
    df_campaign_mix_s2,
    xvar="value_tier:N",
    yvar="risk_level:N",
    textvar="expected_savings_per_customer:Q",
    xsort=["Bronze", "Silver", "Gold", "Platinum"],
    ysort=["High", "Medium", "Low"],
    color_by_col="expected_savings_per_customer:Q",
    legend_title="Savings per Customer",
    border_attrs=dict(stroke="white", strokeWidth=1.0),
    scale_params=dict(scheme="reds"),
    text_fontsize=18,
    text_alt_condition=text_alt_condition_s2,
    tooltip=tooltip_s2,
    ptitle=ptitle_s2,
    xtitle="Revenue Tier (CLV)",
    ytitle="Churn Risk Level",
    fig_size=dict(width=350, height=350),
    save_params=dict(
        fpath=figures_dir / "fig_30_cohort_savings_hmap_cs2.html"
    ),
)
chart_budget

#### Comparison

Below is a summary of the business metrics for the two scenarios

In [ ]:
(
    pd.concat(
        [
            df_campaign_mix_s1.assign(scenario=1),
            df_campaign_mix_s2.assign(scenario=2),
        ]
    )
    .groupby("scenario", as_index=False)
    .agg(
        {
            "expected_savings": "sum",
            "true_savings": "sum",
            "num_customers": "sum",
            "total_intervention_cost": "sum",
        }
    )
    .assign(
        expected_savings_per_customer=lambda df: df["expected_savings"].div(
            df["num_customers"]
        ),
        savings_error_pct=lambda df: (
            df["expected_savings"]
            .sub(df["true_savings"])
            .div(df["true_savings"])
            .mul(100)
        ),
    )
    .style.set_properties(
        subset=["savings_error_pct", "expected_savings"],
        **{"background-color": "yellow", "color": "black"},
    )
)

The errors in the expected savings, relative to the true savings, are similar for both scenarios. We will consider them both negligible as they are both below 5%. The error for scenario 2 is even lower, below 2%.

In terms of expected savings, secnario 2 is the best but this comes at the cost of selecting more customers. Correcting for this, by using expected savings per customer, shows that scenario 1 is the better of the two scenarios.

Scenario 2 delivers the higher expected savings per customer. Scenario 1 gives the higher predicted savings.

### Append Metadata to Business Metrics

For convenience, before exporting to disk, we will append a column with a copy of `y_pred` renamed to `is_at_risk` since it indicates if a customer is at-risk (1) or not (0)

In [ ]:
%%time
df_ranked_s1_s2 = pd.concat(
    [df_ranked_s1.assign(scenario=1), df_ranked_s2.assign(scenario=2)]
).assign(is_at_risk=lambda df: df["y_pred"])
_ = show_df(df_ranked_s1_s2)

## Export Project Deliverables to Private R2 Bucket

Get the current timestamp in the format `YYmmdd_HHMMSS`

In [ ]:
curr_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

Next, export to a file in the R2 bucket with the following file name format `at_risk_customers_with_business_metrics__<best-model-name>__<current-timestamp-YYmmdd_HHMMSS>.parquet.gzip`

In [ ]:
%%time
key_prefix = r2_key_pred.split("/")[0]
r2io.export_df_to_r2(
    s3_client=s3_client,
    df=df_ranked_s1_s2,
    bucket_name=bucket_name,
    r2_key=(
        f"{key_prefix}/at_risk_customers_with_business_metrics_savings__"
        f"{best_model_name.lower()}__"
        f"{curr_timestamp}.parquet.gzip"
    ),
    verbose=False,
)

## Conclusion

Our analysis has identified nine profitable customer segments by combining Value Tiers with Risk Levels. We recommend excluding all *Bronze* tier customers from intervention, as these segments are unlikely to yield a positive return.

We evaluated two primary targeting scenarios based on budget availability:

1. Scenario 1 (High Efficiency)
   - By targeting the top 400 customers (requiring a 20,000 dollar budget), we estimate 150,000 dollars in total savings. This approach focuses exclusively on the five highest-performing segments, delivering an average benefit of 376 dollars per customer.
2. Scenario 2 (Maximum Reach)
   - By targeting all 1,600 profitable customers (requiring an 80,000 dollars budget), estimated savings increase to 205,000 dollars.

Scenario 1 is the superior choice for savings. It captures the bulk of available savings with 43% higher per-customer efficiency than the full-reach approach.

In order for these estimates to be relevant to other customers, not included in the random sample we used in this project, the other sample must have the same characteristics as those seen in the customers whose data was used in the analysis here. In a later notebook, we develop a data validation model that can be used to validate customer data before being used with the model developed here.